# **ENVIROMENT INITIALIZATION**

In [1]:
import polars as pl

# CONFIGURATION
from config import (
    PROJECT_ROOT,
    AS_OF, TARGET_START, TARGET_END,
    DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_FEATURES_DIR,
    TRAIN_PATH, CV_TARGET_PATH, HISTORY_PATH, CV_FEATURES_PATH,
    REPORTS_DIR, REPORTS_GMV_DIR, REPORTS_CONV_FUNL_DIR, REPORTS_FEATURES_DIR,
    REPORTS_MODELS_DIR, REPORTS_MODELS_GRAPHS_DIR,
    BIN_ORDER,
    RANDOM_STATE,
    create_directories,
    validate_data_exists,
)

# Creating directories
create_directories()

# Checking for data availability
validate_data_exists()

All directories created successfully
TRAIN_PATH: 171.80 MB

All data files are present


True

# **DATA COMPOSING**

In [2]:
# DATA LOADING
train_lf = pl.scan_parquet(TRAIN_PATH)

# Sanity check
train_info = train_lf.select(
    pl.col("event_date").min().alias("min_date"),
    pl.col("event_date").max().alias("max_date"),
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("users"),
).collect()

print(f"Train data loaded: {train_lf.collect_schema()}")
print(train_info)

Train data loaded: Schema({'event_date': Date, 'user_id': Int64, 'search': Int64, 'cat': Int64, 'has_search_to_cart': Int64, 'has_search_to_ord': Int64, 'has_cat_to_cart': Int64, 'has_cat_to_ord': Int64, 'search_to_cart': Int64, 'search_to_ord': Int64, 'cat_to_cart': Int64, 'cat_to_ord': Int64, 'gmv_search': Float64, 'gmv_cat': Float64, 'to_cart': Int64, 'to_ord': Int64, 'gmv': Float64, 'searches': Int64})
shape: (1, 4)
┌────────────┬────────────┬──────────┬────────┐
│ min_date   ┆ max_date   ┆ rows     ┆ users  │
│ ---        ┆ ---        ┆ ---      ┆ ---    │
│ date       ┆ date       ┆ u32      ┆ u32    │
╞════════════╪════════════╪══════════╪════════╡
│ 2025-01-01 ┆ 2026-02-13 ┆ 30631006 ┆ 250000 │
└────────────┴────────────┴──────────┴────────┘


In [3]:
# CREATE HISTORY SNAPSHOT
# Filtering only the data up to AS_OF
history_lf = train_lf.filter(pl.col("event_date") <= AS_OF)

# Sanity check
history_info = history_lf.select(
    pl.col("event_date").min().alias("min_date"),
    pl.col("event_date").max().alias("max_date"),
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("users"),
).collect()

print("History snapshot info:")
print(history_info)

history_lf.sink_parquet(HISTORY_PATH)
print(f"\nHistory saved to: {HISTORY_PATH}")
print(f"File size: {HISTORY_PATH.stat().st_size / 1024 / 1024:.2f} MB")

History snapshot info:
shape: (1, 4)
┌────────────┬────────────┬──────────┬────────┐
│ min_date   ┆ max_date   ┆ rows     ┆ users  │
│ ---        ┆ ---        ┆ ---      ┆ ---    │
│ date       ┆ date       ┆ u32      ┆ u32    │
╞════════════╪════════════╪══════════╪════════╡
│ 2025-01-01 ┆ 2026-01-14 ┆ 27837039 ┆ 250000 │
└────────────┴────────────┴──────────┴────────┘

History saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\history_before_2026-01-14.parquet
File size: 158.26 MB


In [4]:
# CREATE TARGET
# Filtering only the data after AS_OF
cv_target_lf = train_lf.filter(
    (pl.col("event_date") > AS_OF) &
    (pl.col("event_date") <= TARGET_END)
)

# Sanity check
future_info = cv_target_lf.select(
    pl.col("event_date").min().alias("min_target_date"),
    pl.col("event_date").max().alias("max_target_date"),
    pl.len().alias("rows_in_target_window"),
    pl.col("user_id").n_unique().alias("users_in_target_window"),
).collect()

print("Target window info:")
print(future_info)

# Aggregating GMV by users
target_lf = (
    cv_target_lf
    .group_by("user_id")
    .agg(pl.col("gmv").sum().alias("target"))
)

# Getting a list of all users
users_lf = train_lf.select(pl.col("user_id").unique())

# Join with zero filling for users without purchases
cv_target_lf = (
    users_lf
    .join(target_lf, on = "user_id", how = "left")
    .with_columns(pl.col("target").fill_null(0.0))
)

cv_target = cv_target_lf.collect()

# Target statistics
cv_target_stats = cv_target_lf.select(
    pl.len().alias("users"),
    pl.col("target").mean().alias("mean"),
    pl.col("target").median().alias("median"),
    pl.col("target").max().alias("max"),
    (pl.col("target") == 0).mean().alias("zero_share"),
    (pl.col("target") > 0).mean().alias("positive_share"),
    pl.col("target").quantile(0.50).alias("q50"),
    pl.col("target").quantile(0.90).alias("q90"),
    pl.col("target").quantile(0.99).alias("q99"),
    pl.col("target").quantile(0.999).alias("q999"),
).collect()

print("\nTarget statistics:")
print(cv_target_stats)

cv_target.write_parquet(CV_TARGET_PATH)
print(f"\nTarget saved to: {CV_TARGET_PATH}")

Target window info:
shape: (1, 4)
┌─────────────────┬─────────────────┬───────────────────────┬────────────────────────┐
│ min_target_date ┆ max_target_date ┆ rows_in_target_window ┆ users_in_target_window │
│ ---             ┆ ---             ┆ ---                   ┆ ---                    │
│ date            ┆ date            ┆ u32                   ┆ u32                    │
╞═════════════════╪═════════════════╪═══════════════════════╪════════════════════════╡
│ 2026-01-15      ┆ 2026-02-13      ┆ 2793967               ┆ 250000                 │
└─────────────────┴─────────────────┴───────────────────────┴────────────────────────┘

Target statistics:
shape: (1, 10)
┌────────┬───────────┬──────────┬────────────┬───┬──────────┬────────────┬────────────┬────────────┐
│ users  ┆ mean      ┆ median   ┆ max        ┆ … ┆ q50      ┆ q90        ┆ q99        ┆ q999       │
│ ---    ┆ ---       ┆ ---      ┆ ---        ┆   ┆ ---      ┆ ---        ┆ ---        ┆ ---        │
│ u32    ┆ f64     